# Surgical Instrument Force & Motion Analysis — 1 · Data processing

**This is the first of two notebooks.** It reads the raw sequence recordings
(`*.igs.mha`), performs registration, and computes **every metric** used by the
analysis, then writes them all to a **single JSON file** (`DATA_JSON`).

The companion notebook **`surgical_force_visualization.ipynb`** reads that JSON and
produces all the figures and tables — no raw data or recomputation required.

```
surgical_force_processing.ipynb   →   analysis_data.json   →   surgical_force_visualization.ipynb
        (read + compute)                 (single file)              (plot everything)
```

Input layout — one zip whose contents are **one subfolder per participant**, each
holding that participant's trial `.igs.mha` files:

```
study.zip
 └─ P01/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
 └─ P02/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
 └─ ...
```

Each file contains, in the header: `Resection_Location` (the fixed resection point for
that whole trial) and `BipolarCollectedPoint0..3` (4 fiducials for registration); and,
per frame: `*TipToWorldTransform` (4×4 pose of each instrument tip — **Bipolar**,
**Cavitron**, **Scissors**), a per-instrument tracking **`…Status`** flag (`OK` /
`INVALID`), `Force` (`fx fy fz tx ty tz`) and `Timestamp` (seconds).

**What this notebook computes** — rigid registration to a common frame, time normalized
to [0, 1], force magnitude & rate of change, velocity/acceleration/jerk, 3D trajectories,
inter-instrument distance & angle, angular speed/acceleration, geometric/economy/
localization metrics and the per-trial statistics table. An instrument counts as **in
use** while its tip is within `RESECTION_DIST_MAX` of `Resection_Location`, and
**tracking success is measured over those in-use frames only**. All of it is serialized
to `DATA_JSON`.

## 1 · Setup

In [ ]:
# Colab already ships numpy / scipy / matplotlib; this is a no-op there and a
# convenience when running elsewhere.
import importlib, subprocess, sys
for pkg in ("numpy", "scipy", "matplotlib", "pandas"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, re, glob, warnings
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# In-use masking leaves all-NaN slices for instruments never in use in a trial;
# nanmean/nanmax on those are expected and return NaN — silence the routine warnings.
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")

# ---- Plot style ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "legend.fontsize": 9, "legend.frameon": False,
    "font.size": 10,
})

INSTRUMENTS = ["Bipolar", "Cavitron", "Scissors"]
# Okabe-Ito colorblind-safe palette, fixed order per instrument.
INST_COLOR = {"Bipolar": "#0072B2", "Cavitron": "#E69F00", "Scissors": "#009E73"}
# Sequential map used everywhere "value = time".
TIME_CMAP = "viridis"
print("Setup complete.")

## 2 · Provide the data as a single zip

Package the whole study as **one `.zip`** whose contents are the participant
subfolders, each holding that participant's trial `.igs.mha` files:

```
study.zip
 └─ P01/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
 └─ P02/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
 └─ ...
```

Set `DATA_ZIP` to that file. The zip is extracted and the participant / trial
structure is recovered automatically.

- An extra wrapper folder inside the zip (e.g. everything under a top-level `data/`)
  is fine — participants are detected by the folder that directly contains the
  `.igs.mha` files, at any depth.
- **Colab** — run the upload cell to pick the zip, or mount Google Drive and point
  `DATA_ZIP` at it.

In [ ]:
DATA_ZIP = "data/study.zip"   # the single zip containing the participant subfolders
WORK_DIR = "data_unzipped"    # where the zip is extracted (recreated on each run)
TIME_UNIT = "s"               # timestamps are in seconds
SMOOTH_WINDOW = 11            # Savitzky-Golay window (odd, in frames) for differentiation; 0 disables
RESECTION_DIST_MAX = 30.0     # mm; an instrument tip closer than this to Resection_Location is "in use"
DATA_JSON = "data/analysis_data.json"  # single metrics file consumed by the visualization notebook

In [ ]:
# --- Optional: upload the zip directly in Colab ---------------------------
# Uncomment in Colab, then pick your study .zip. It is saved to DATA_ZIP.
#
# from google.colab import files
# os.makedirs(os.path.dirname(DATA_ZIP) or ".", exist_ok=True)
# up = files.upload()
# name = next(n for n in up if n.lower().endswith(".zip"))
# with open(DATA_ZIP, "wb") as f:
#     f.write(up[name])
# print("saved", name, "->", DATA_ZIP)
#
# --- Optional: mount Google Drive instead ---------------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_ZIP = "/content/drive/MyDrive/study.zip"

In [ ]:
import zipfile, shutil

assert os.path.isfile(DATA_ZIP), (
    f"Zip not found: {DATA_ZIP!r}. Upload it (see the cell above) or fix DATA_ZIP.")

# Fresh extraction every run so stale files never linger.
if os.path.isdir(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as z:
    z.extractall(WORK_DIR)
print(f"Extracted {DATA_ZIP} -> {WORK_DIR}/")


def discover_participants(root):
    "Recursively find trials; participant id = folder that directly holds the .igs.mha files."
    parts = {}
    for f in sorted(glob.glob(os.path.join(root, "**", "*.igs.mha"), recursive=True)):
        if "__MACOSX" in f:                       # skip macOS zip cruft
            continue
        pid = os.path.basename(os.path.dirname(f))
        parts.setdefault(pid, []).append(f)
    return {pid: sorted(fs) for pid, fs in parts.items()}

participant_files = discover_participants(WORK_DIR)
assert participant_files, (
    f"No .igs.mha files inside {DATA_ZIP!r}. Expected participant subfolders "
    "like P01/trial1.igs.mha.")

print(f"\nFound {len(participant_files)} participant(s):")
for pid, fs in participant_files.items():
    print(f"  {pid}: {len(fs)} trial(s)")
    for f in fs:
        print(f"      {os.path.basename(f)}")

## 3 · Parse the sequence files (tracking status + in-use detection)

The `.igs.mha` header is a flat list of `Key = Value` lines; per-frame fields are
named `Seq_Frame<idx>_<Field>`. We read tracking, force, timestamps, the fiducial
points and the trial's **`Resection_Location`** — the referenced video is ignored. Every
trial is tagged with its participant.

**Tracking status.** Each instrument carries a per-frame status field
(`Seq_Frame<idx>_<Instrument>Status`, falling back to
`Seq_Frame<idx>_<Instrument>TipToWorldTransformStatus`) whose value is `OK` or `INVALID`.
It is read straight into a per-instrument **`tracking_status`** array: `1` = tracked
(`OK`), `0` = not tracked (anything else). Older recordings that carry no status field at
all fall back to the legacy heuristic — a pose bit-for-bit identical to the previous
frame means the instrument was not tracked.

**In use.** `Resection_Location` is a **single fixed point for the whole trial**. Per
frame we compute the distance from each instrument tip to it; an instrument is **in use**
while that distance is below `RESECTION_DIST_MAX` (30 mm), and parked / not in use
otherwise. This proximity gate replaces the former working-area gate (distance to the
fiducial plane + inside the fiducial polygon). A rigid transform preserves distance, so
the gate is identical before and after registration.

**Tracking success is reported over in-use frames only** — whether an instrument was
tracked while parked away from the resection site does not matter for the analysis.

In [ ]:
# Header keys that may hold the trial's fixed resection point, in order of preference.
RESECTION_KEYS = ("Resection_Location", "ResectionLocation", "ResectionPoint")


def _parse_point(s):
    "'x y z' (space- or comma-separated) -> (3,) array; None if it doesn't hold 3 numbers."
    if s is None:
        return None
    try:
        vals = [float(x) for x in re.split(r"[,\s]+", s.strip()) if x]
    except ValueError:
        return None
    return np.array(vals[:3], float) if len(vals) >= 3 else None


def _status_key(frame, inst):
    "Per-frame tracking-status field name for `inst` in this file, or None if it has none."
    for k in (f"{inst}Status", f"{inst}TipToWorldTransformStatus", f"{inst}ToTrackerTransformStatus"):
        if k in frame:
            return k
    return None


def parse_mha(path, participant, trial_no):
    header, frames = {}, {}
    with open(path, "r", errors="ignore") as fh:
        for line in fh:
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip()
            m = re.match(r"Seq_Frame(\d+)_(.+)", k)
            if m:
                frames.setdefault(int(m.group(1)), {})[m.group(2)] = v
            else:
                header[k] = v
            if k == "ElementDataFile":   # end of header; pixel data (if any) follows
                break

    idx = sorted(frames)

    def as_mat(fr, key):
        return np.array([float(x) for x in fr[key].split()], float).reshape(4, 4)

    n_pts = int(header.get("BipolarCollectedPointCount", 0))
    fiducials = (np.array([[float(x) for x in header[f"BipolarCollectedPoint{i}"].split()]
                           for i in range(n_pts)]) if n_pts else None)

    # Fixed resection point for the whole trial (same tracker/world frame as the poses).
    resection = next((_parse_point(header[k]) for k in RESECTION_KEYS if k in header), None)

    timestamps = np.array([float(frames[i]["Timestamp"]) for i in idx])
    force = np.array([[float(x) for x in frames[i]["Force"].split()] for i in idx])  # fx fy fz tx ty tz

    poses = {}
    for inst in INSTRUMENTS:
        key = inst + "TipToWorldTransform"
        if key in frames[idx[0]]:
            poses[inst] = np.stack([as_mat(frames[i], key) for i in idx])            # (N,4,4)

    # Tracking status: the per-frame "<Instrument>Status" field is OK / INVALID.
    # tracking_status -> 1 = tracked (OK), 0 = not tracked. Recordings without any status
    # field fall back to the legacy rule (a pose repeated bit-for-bit = frozen = not tracked).
    tracking_status = {}
    for inst, M in poses.items():
        skey = _status_key(frames[idx[0]], inst)
        if skey is not None:
            tracking_status[inst] = np.array(
                [1 if frames[i].get(skey, "").strip().upper() == "OK" else 0 for i in idx], int)
        else:
            st = np.ones(len(M), dtype=int)                 # frame 0 has no predecessor -> tracked
            st[1:][np.all(M[1:] == M[:-1], axis=(1, 2))] = 0
            tracking_status[inst] = st

    # In-use detection: distance from each tip to the trial's fixed Resection_Location.
    # Below RESECTION_DIST_MAX the instrument is in use, otherwise it is parked. Distances
    # are rigid-transform invariant, so this is the same before and after registration (§4).
    resection_dist, in_use = {}, {}
    for inst, M in poses.items():
        if resection is None:
            resection_dist[inst] = np.full(len(M), np.nan)
            in_use[inst] = np.ones(len(M), bool)            # no resection point -> no gate
        else:
            d = np.linalg.norm(M[:, :3, 3] - resection, axis=1)
            resection_dist[inst] = d
            in_use[inst] = d < RESECTION_DIST_MAX

    return dict(participant=participant, trial=trial_no,
                name=os.path.basename(path).replace(".igs.mha", ""),
                label=f"{participant}·T{trial_no}",
                header=header, timestamps=timestamps, force=force,
                poses=poses, tracking_status=tracking_status, fiducials=fiducials,
                resection=resection, resection_dist=resection_dist, in_use=in_use)


trials = []                       # flat list of all trials, in participant/trial order
participants = {}                 # participant_id -> list of its trials
for pid, fs in participant_files.items():
    participants[pid] = []
    for t_no, f in enumerate(fs, start=1):
        tr = parse_mha(f, pid, t_no)
        trials.append(tr)
        participants[pid].append(tr)

missing_res = [tr["label"] for tr in trials if tr["resection"] is None]
if missing_res:
    print(f"WARNING: no Resection_Location in {len(missing_res)} trial(s) "
          f"({', '.join(missing_res)}) — every frame is treated as in use there.\n")

for tr in trials:
    dur = tr["timestamps"][-1] - tr["timestamps"][0]
    # per instrument: share of frames in use, and tracking success over those in-use frames
    cells = []
    for inst in tr["poses"]:
        u = tr["in_use"][inst]
        ok = 100 * tr["tracking_status"][inst][u].mean() if u.any() else np.nan
        cells.append(f"{inst}:{100 * u.mean():5.1f}% in use / {ok:5.1f}% tracked")
    print(f"{tr['label']:<12s} {len(tr['timestamps']):5d} frames {dur:6.1f} {TIME_UNIT}  "
          + "  ".join(cells))

In [ ]:
# ---- Tracking report: tracking success over IN-USE frames only ---------------------
# An instrument parked away from the resection site does not need to be tracked, so the
# tracking figure only counts frames where the tip is within RESECTION_DIST_MAX of
# Resection_Location. Frame-weighted per participant, and overall across all participants.
def _inuse_stats(trial_list, inst):
    "(% of frames in use, % of those in-use frames that are tracked) or (None, None)."
    use = ok = tot = 0
    for tr in trial_list:
        if inst in tr["tracking_status"]:
            u = tr["in_use"][inst]
            use += int(u.sum()); tot += int(u.size)
            ok += int((tr["tracking_status"][inst][u] == 1).sum())
    if not tot:
        return None, None
    return 100 * use / tot, (100 * ok / use if use else None)


def _report(title, pick):
    print(title + "\n")
    head = f"{'participant':<13s}" + "".join(f"{inst:>12s}" for inst in INSTRUMENTS)
    print(head); print("-" * len(head))
    for label, tlist in list(participants.items()) + [("ALL", trials)]:
        if label == "ALL":
            print("-" * len(head))
        cells = []
        for inst in INSTRUMENTS:
            p = pick(_inuse_stats(tlist, inst))
            cells.append(f"{p:11.2f}%" if p is not None else f"{'—':>12s}")
        print(f"{label:<13s}" + "".join(cells))
    print()


_report("Instrument in use  (% of frames within "
        f"{RESECTION_DIST_MAX:.0f} mm of Resection_Location)", lambda s: s[0])
_report("Tracking success  (% of IN-USE frames with status OK)", lambda s: s[1])

## 4 · Point-wise registration to a common reference frame

Every trial carries the **same 4 physical fiducials** (`BipolarCollectedPoint0..3`).
We compute the rigid transform (rotation + translation, no scaling) that best maps
each trial's fiducials onto a single **reference set** — by default the first trial
of the first participant — using the closed-form SVD solution (Kabsch / Horn's
absolute orientation). Applying that transform to every `*TipToWorldTransform`
position — and to the trial's `Resection_Location` — places **all trials of all
participants** in one shared frame, so trajectories are directly comparable.

The per-trial fiducial **RMSE** flags a mislabeled or mis-collected point.

We also build, once, the **working-plane frame** used later for the force pyramid: the
plane through the 4 reference fiducials, with orthonormal in-plane axes `(u, v)` from
their SVD and the **out-of-plane normal `n`** as the third axis. Each trial's pyramid is
centered on its own registered `Resection_Location` (§8 of the visualization notebook).

In [ ]:
def rigid_register(src, dst):
    "Least-squares rigid transform (4x4) mapping src points onto dst; returns (G, rmse)."
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    cs, cd = src.mean(0), dst.mean(0)
    H = (src - cs).T @ (dst - cd)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))          # guard against reflection
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    t = cd - R @ cs
    G = np.eye(4); G[:3, :3] = R; G[:3, 3] = t
    rmse = np.sqrt(np.mean(np.sum(((R @ src.T).T + t - dst) ** 2, axis=1)))
    return G, rmse


def _plane_frame(pts):
    """Working-plane frame of >=3 points: (centroid, u, v, n).

    `u, v` are the in-plane principal directions (SVD of the centered points, so they
    follow the working rectangle's sides) and `n` is the **out-of-plane normal** — the
    depth axis the force pyramid sums along, and the plane's z direction."""
    pts = np.asarray(pts, float)
    c = pts.mean(0)
    _, _, vt = np.linalg.svd(pts - c)               # rows: principal axes, last = normal
    return c, vt[0], vt[1], vt[2]


ref_pts = trials[0]["fiducials"]                    # common reference for every trial

for tr in trials:
    if tr["fiducials"] is not None and ref_pts is not None:
        tr["G"], tr["rmse"] = rigid_register(tr["fiducials"], ref_pts)
    else:
        tr["G"], tr["rmse"] = np.eye(4), np.nan

    tr["pos"] = {}
    for inst, P in tr["poses"].items():
        xyz = P[:, :3, 3]
        tr["pos"][inst] = (tr["G"][:3, :3] @ xyz.T).T + tr["G"][:3, 3]

    # the trial's fixed resection point, in the common registered frame
    tr["resection_reg"] = (None if tr["resection"] is None
                           else tr["G"][:3, :3] @ tr["resection"] + tr["G"][:3, 3])

    t = tr["timestamps"]
    tr["tnorm"] = (t - t[0]) / (t[-1] - t[0])       # duration normalized to [0, 1]

    print(f"{tr['label']:<12s} registration RMSE = {tr['rmse']:.3f} mm")

# ---- working-plane frame (shared by every trial), from the reference fiducials -------
# Orientation only: in-plane axes u, v and the out-of-plane normal n. The force pyramid
# is centered per trial on its own registered Resection_Location, not on this centroid.
if ref_pts is not None and len(ref_pts) >= 3:
    PLANE_C, PLANE_U, PLANE_V, PLANE_N = _plane_frame(ref_pts)
    print(f"\nworking plane   normal = [{PLANE_N[0]:+.4f} {PLANE_N[1]:+.4f} {PLANE_N[2]:+.4f}]  "
          f"(from {len(ref_pts)} reference fiducials)")
else:
    PLANE_C = PLANE_U = PLANE_V = PLANE_N = None
    print("\nno fiducials — no working-plane frame (the force pyramid falls back to tracker x/y/z)")

## 5 · Derived signals and metrics

> **In-use masking (applied to every tracking-derived signal).** Per instrument, per
> frame, we build a validity mask and set all tracking-derived data to **NaN** outside it
> — so masked frames are neither plotted nor averaged:
> - **Every instrument** (Bipolar, Cavitron, Scissors): **in use** while its tip is within
>   `RESECTION_DIST_MAX = 30 mm` of the trial's fixed **`Resection_Location`** (§3). A tip
>   farther than that is parked away from the resection site and not in use. This single
>   proximity rule replaces the previous working-area gate (distance to the fiducial plane
>   plus inside the fiducial polygon).
> - Signals additionally require **valid tracking** (`Status = OK`, §3): an `INVALID` frame
>   carries no usable pose, so nothing can be derived from it. Tracking success itself is
>   reported over **in-use frames only** (`tr["track_ok_inuse"]`).
>
> This masks velocity, acceleration, jerk, angular speed, angular acceleration, the 3D
> trajectory, the
> inter-instrument distance and angle, and the geometric metrics (path length, working
> volume, straightness, idle fraction). **Force** comes from a separate sensor and is
> *not* masked.
>
> The **fiducial plane** is still computed (§4) — but only for its *orientation*: its
> out-of-plane normal is the z direction of the force-pyramid map, which is centered on
> `Resection_Location`.

**Signals over time**

- **Force magnitude** `|F| = √(fx² + fy² + fz²)` (invariant to registration).
- **Distance to the resection site** `‖p(t) − Resection_Location‖` per instrument — the
  quantity the in-use flag thresholds at 30 mm.
- **Velocity, acceleration, jerk** — 1st/2nd/3rd time-derivatives of tip position,
  magnitudes reported. Position is lightly Savitzky-Golay smoothed before
  differentiating (numerical differentiation amplifies tracking noise); derivatives
  use the **actual timestamps**, so units are mm/s, mm/s², mm/s³.
- **Inter-instrument distance** `dₐᵦ(t) = ‖pₐ(t) − pᵦ(t)‖` for the pairs meant to be
  used together — **Bipolar–Cavitron** and **Bipolar–Scissors** (Cavitron–Scissors is
  *not* used together and is excluded). It uses the same §5 in-use masking as every other
  tracking metric: frames where either instrument is not in use are set to **NaN** and
  neither plotted nor averaged. Rigid registration preserves distances, so these are
  unaffected by it.
- **Inter-instrument angle** `θₐᵦ(t)` between the two instruments' **long axes** (see
  below), for the same pairs. Set to NaN on frames where either instrument is untracked.

**Instrument long axis (orientation).** The `*TipToWorldTransform` is a *pivot*
calibration: the tip is the frame origin and the `*TipTo*Transform` translation `d` is
the tip position in the tool-marker frame. The shaft therefore runs along the pivot
direction — the line from the tip back to the marker — **not** along a coordinate axis
of the tip frame (for these tools the closest tip axis is only ~0.83–0.91 aligned). We
take the long axis as `â = normalize(−Rᵀd)` in the tip frame and rotate it to world each
frame with the pose's rotation. Angular quantities (inter-instrument angle, angular
speed) follow from it.

**Scalar metrics per trial / instrument**

- **Path length** `L = Σ‖Δp‖` — total distance the tip travels (mm).
- **Angular speed** — rotational speed of the instrument's frame, deg/s *(suggested)*.
- **Angular acceleration** — rate of change of angular speed, deg/s² *(suggested)*.
- **Net displacement** `‖p_end − p_start‖` and **straightness** `= net / L ∈ (0, 1]`
  (economy of motion; 1 = perfectly straight). *(suggested)*
- **Working volume** — volume of the tip's axis-aligned bounding box, mm³ *(suggested)*.
- **Idle fraction** — share of time the tip moves slower than `IDLE_SPEED` *(suggested)*.
- **Force impulse** `∫|F| dt` and **peak / mean force**, **mean torque** *(suggested)*.
- **Force coefficient of variation** `CoV = SD(|F|) / mean(|F|) × 100%` — normalised force variability; lower = steadier force control *(suggested)*.
- **Mean absolute rate of force change** `⟨|dF/dt|⟩` (N/s) — how fast `|F|` changes frame-to-frame, using the actual timestamps; lower = more controlled, gradual force application (distinct from impulse, which is the time-integral of force) *(suggested)*.

The *(suggested)* metrics are ones I added as commonly-used surgical-dexterity
indicators — keep or drop them as you like.

In [ ]:
def _smooth(x, win):
    if not win or win < 3 or win >= len(x):
        return x
    if win % 2 == 0:
        win += 1
    try:
        from scipy.signal import savgol_filter
        return savgol_filter(x, win, 3, axis=0)
    except Exception:                     # fallback: centered moving average
        k = np.ones(win) / win
        return np.stack([np.convolve(x[:, j], k, mode="same") for j in range(x.shape[1])], 1)


def kinematics(pos, t, win):
    pos = _smooth(pos, win)
    vel = np.gradient(pos, t, axis=0)
    acc = np.gradient(vel, t, axis=0)
    jrk = np.gradient(acc, t, axis=0)
    return {"velocity": np.linalg.norm(vel, axis=1),
            "acceleration": np.linalg.norm(acc, axis=1),
            "jerk": np.linalg.norm(jrk, axis=1)}


# Instrument pairs that are meant to be used together (Cavitron+Scissors are not).
PAIRS = [("Bipolar", "Cavitron"), ("Bipolar", "Scissors")]
PAIR_KEYS = [f"{a}-{b}" for a, b in PAIRS]
PAIR_COLOR = {"Bipolar-Cavitron": "#7A5195", "Bipolar-Scissors": "#EF5675"}
IDLE_SPEED = 5.0        # mm/s; frames slower than this count as idle/dwell (suggested metric)
VOXEL_SIZE = 2.0        # mm; voxel edge for the occupancy-entropy (spatial concentration) metric

def _path_length(P):
    return float(np.sum(np.linalg.norm(np.diff(P, axis=0), axis=1)))

def _trapz(y, x):                                                    # version-independent ∫y dx
    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))

def long_axis_tip(header, inst):
    "Instrument long (shaft) axis as a unit vector in its calibrated Tip frame."
    key = f"{inst}TipTo{inst}Transform"
    if key not in header:
        return None
    T = np.array([float(x) for x in header[key].split()], float).reshape(4, 4)
    R, d = T[:3, :3], T[:3, 3]        # d = tip position in the tool-marker frame (pivot offset)
    a = -R.T @ d                      # direction tip -> marker, expressed in the Tip frame
    n = np.linalg.norm(a)
    return a / n if n > 0 else None

def angular_speed(R_stack, t):
    "Per-frame rotational speed (deg/s) from the relative rotation between frames."
    dR = np.matmul(R_stack[1:], np.transpose(R_stack[:-1], (0, 2, 1)))
    cos = np.clip((np.trace(dR, axis1=1, axis2=2) - 1) / 2, -1, 1)
    sp = np.degrees(np.arccos(cos)) / np.diff(t)
    return np.concatenate([sp[:1], sp])            # pad to length N

for tr in trials:
    t = tr["timestamps"]
    ts = tr["tracking_status"]
    n = len(t)
    tr["fmag"] = np.linalg.norm(tr["force"][:, :3], axis=1)          # |force| (force sensor, not masked)
    tr["tmag"] = np.linalg.norm(tr["force"][:, 3:], axis=1)          # |torque|
    tr["impulse"] = _trapz(tr["fmag"], t)                            # force impulse ∫|F|dt
    fmean = float(tr["fmag"].mean())
    tr["force_cov"] = float(tr["fmag"].std() / fmean * 100.0) if fmean > 0 else np.nan   # SD/mean×100 (%)
    tr["dFdt"] = np.abs(np.gradient(tr["fmag"], t))                                       # |dF/dt| per frame (N/s)
    tr["dFdt_abs_mean"] = float(np.mean(tr["dFdt"]))                                       # mean |dF/dt| (N/s)

    # raw tip-tip distances (before any masking) — used by the inter-instrument distance metric
    raw_dist = {f"{a}-{b}": np.linalg.norm(tr["pos"][a] - tr["pos"][b], axis=1)
                for a, b in PAIRS if a in tr["pos"] and b in tr["pos"]}

    # ---- per-instrument "in use" validity mask -------------------------------------
    # In use (§3) = tip within RESECTION_DIST_MAX of the trial's fixed Resection_Location.
    # Deriving a signal additionally needs a valid pose, so the mask applied to every
    # tracking-derived signal below is (in use AND Status == OK); everything else -> NaN.
    valid = {inst: (tr["in_use"][inst] & (ts[inst] == 1)) for inst in tr["pos"]}
    tr["valid"] = valid
    tr["inuse_frac"] = {inst: float(tr["in_use"][inst].mean()) for inst in tr["pos"]}
    # tracking success counts only in-use frames: an instrument parked away from the
    # resection site does not need to be tracked.
    tr["track_ok_inuse"] = {
        inst: (float((ts[inst][tr["in_use"][inst]] == 1).mean())
               if tr["in_use"][inst].any() else np.nan)
        for inst in tr["pos"]}

    # ---- kinematics: computed on the full track, then masked to in-use frames ------
    tr["kin"] = {}
    for inst in tr["pos"]:
        k = kinematics(tr["pos"][inst], t, SMOOTH_WINDOW)
        tr["kin"][inst] = {m: np.where(valid[inst], arr, np.nan) for m, arr in k.items()}

    # ---- long-axis orientation + angular speed & acceleration (masked) --------------
    tr["axis"], tr["ang_speed"], tr["ang_accel"] = {}, {}, {}
    for inst in tr["poses"]:
        a_tip = long_axis_tip(tr["header"], inst)
        Rw = tr["poses"][inst][:, :3, :3]                # tip-frame orientation in world, per frame
        tr["axis"][inst] = (Rw @ a_tip) if a_tip is not None else None
        v = valid.get(inst, np.ones(n, bool))
        sp = angular_speed(Rw, t)                         # angular speed (deg/s)
        # smooth the speed before differentiating (numerical differentiation amplifies
        # noise), mirroring the position smoothing used for the linear kinematics
        sp_s = _smooth(sp[:, None], SMOOTH_WINDOW)[:, 0]
        acc = np.gradient(sp_s, t)                        # angular acceleration (deg/s²)
        tr["ang_speed"][inst] = np.where(v, sp, np.nan)
        tr["ang_accel"][inst] = np.where(v, acc, np.nan)

    # ---- geometric / economy metrics over in-use frames only -----------------------
    tr["pathlen"], tr["netdisp"], tr["straightness"] = {}, {}, {}
    tr["bbox_vol"], tr["idle_frac"] = {}, {}
    # localization metrics: orientation-invariant spread + dwell concentration
    tr["ell_vol"], tr["aniso"], tr["occ_eff"], tr["occ_voxels"] = {}, {}, {}, {}
    dt = np.gradient(t)                               # per-frame dwell duration (s), for time weighting
    for inst, P in tr["pos"].items():
        v = valid[inst]
        idx = np.flatnonzero(v)
        if idx.size >= 2:
            Pv = P[v]                                     # in-use tip positions (registered frame)
            seg_ok = v[1:] & v[:-1]                       # count only steps between two in-use frames
            L = float(np.sum(np.linalg.norm(np.diff(P, axis=0), axis=1)[seg_ok]))
            net = float(np.linalg.norm(P[idx[-1]] - P[idx[0]]))
            bbox = float(np.prod(np.ptp(Pv, axis=0)))     # axis-aligned working volume (kept as-is)
            idle = float(np.mean(tr["kin"][inst]["velocity"][v] < IDLE_SPEED))

            # covariance-ellipsoid volume: orientation-invariant spread (frame-independent).
            # eigenvalues of the 3x3 position covariance are variances (mm^2) along the cloud's
            # own principal axes; the 1-sigma ellipsoid has semi-axes sqrt(lam) -> V=(4/3)pi*prod(sqrt(lam)).
            cov = np.cov(Pv, rowvar=False)
            lam = np.clip(np.sort(np.linalg.eigvalsh(cov))[::-1], 0.0, None)   # lam1>=lam2>=lam3
            ell = float((4.0 / 3.0) * np.pi * np.sqrt(np.prod(lam)))           # mm^3
            aniso = float(lam[0] / lam[2]) if lam[2] > 0 else np.nan           # variance ratio: line/plane vs blob

            # occupancy entropy: dwell-time-weighted concentration over VOXEL_SIZE voxels.
            # effective #occupied voxels exp(H); low -> time spent revisiting a tight core.
            w = dt[v]
            keys = np.floor(Pv / VOXEL_SIZE).astype(np.int64)
            _, inv = np.unique(keys, axis=0, return_inverse=True)
            inv = np.asarray(inv).ravel()
            wsum = np.zeros(inv.max() + 1)
            np.add.at(wsum, inv, w)
            wsum = wsum[wsum > 0]
            if wsum.sum() > 0:
                p = wsum / wsum.sum()
                H = float(-np.sum(p * np.log(p)))
                occ_eff = float(np.exp(H))
                occ_vox = int(p.size)
            else:
                occ_eff, occ_vox = np.nan, 0
        else:
            L = net = bbox = idle = ell = aniso = occ_eff = np.nan
            occ_vox = 0
        tr["pathlen"][inst] = L
        tr["netdisp"][inst] = net
        tr["straightness"][inst] = (net / L) if (L and L > 0) else np.nan
        tr["bbox_vol"][inst] = bbox
        tr["idle_frac"][inst] = idle
        tr["ell_vol"][inst] = ell
        tr["aniso"][inst] = aniso
        tr["occ_eff"][inst] = occ_eff
        tr["occ_voxels"][inst] = occ_vox

    # mean tip-to-resection distance over the frames the instrument is in use
    tr["resection_dist_mean"] = {
        inst: (float(np.mean(tr["resection_dist"][inst][valid[inst]]))
               if valid[inst].any() else np.nan)
        for inst in tr["pos"]}

    # ---- inter-instrument distance: NaN where either instrument is not in use ----------
    tr["dist"] = {}
    for a, b in PAIRS:
        key = f"{a}-{b}"
        if key in raw_dist:
            d = raw_dist[key].astype(float).copy()
            d[(~valid[a]) | (~valid[b])] = np.nan   # §5 in-use mask (near the resection site + tracked)
            tr["dist"][key] = d

    # ---- inter-instrument angle: NaN where either instrument is not in use ----------
    tr["angle"] = {}
    for a, b in PAIRS:
        key = f"{a}-{b}"
        if tr["axis"].get(a) is not None and tr["axis"].get(b) is not None:
            cos = np.clip(np.sum(tr["axis"][a] * tr["axis"][b], axis=1), -1, 1)
            ang = np.degrees(np.arccos(cos))
            ang[(~valid[a]) | (~valid[b])] = np.nan
            tr["angle"][key] = ang

    # ---- masked positions for trajectory plots (in-use frames only) ----------------
    tr["pos_plot"] = {inst: np.where(valid[inst][:, None], P, np.nan)
                      for inst, P in tr["pos"].items()}

KIN_UNITS = {"velocity": "mm/s", "acceleration": "mm/s²", "jerk": "mm/s³"}
# a light->dark ramp to distinguish the trials within one participant
def trial_colors(n):
    return plt.cm.cividis(np.linspace(0.15, 0.85, max(n, 1)))
print("Derived force, kinematics, geometry, inter-instrument distance and orientation metrics.")

## Per-trial statistics table

One row per trial with all summary metrics, plus a per-participant aggregate (mean
across each participant's trials). Both tables are written to CSV next to the zip and
also carried inside the exported JSON so the visualization notebook can display them.

All per-instrument and pair columns reflect the **in-use masking** (frames where the tip
is farther than 30 mm from `Resection_Location`, or where tracking status is `INVALID`,
are excluded). `<inst>_inuse_pct` is the share of frames the instrument was at the
resection site, and `<inst>_tracked_pct` is the tracking success **over those in-use
frames only**; `<inst>_resect_dist_mm` is the mean tip-to-resection distance while in use.

In [ ]:
import pandas as pd

def _nanmean(a):
    a = np.asarray(a, float)
    return float(np.nanmean(a)) if np.any(~np.isnan(a)) else np.nan

def _nanmax(a):
    a = np.asarray(a, float)
    return float(np.nanmax(a)) if np.any(~np.isnan(a)) else np.nan

def _r(x, nd):
    x = float(x)
    return round(x, nd) if np.isfinite(x) else np.nan

rows = []
for tr in trials:
    row = {
        "participant": tr["participant"],
        "trial": tr["trial"],
        "file": tr["name"],
        "n_frames": len(tr["timestamps"]),
        "duration_s": round(float(tr["timestamps"][-1] - tr["timestamps"][0]), 3),
        "reg_rmse_mm": round(float(tr["rmse"]), 4),
        "force_mean_N": round(float(tr["fmag"].mean()), 5),
        "force_peak_N": round(float(tr["fmag"].max()), 5),
        "force_cov_pct": _r(tr["force_cov"], 3),
        "force_impulse": round(float(tr["impulse"]), 4),
        "force_dFdt_Nps": round(float(tr["dFdt_abs_mean"]), 4),
        "torque_mean": round(float(tr["tmag"].mean()), 5),
    }
    for inst in INSTRUMENTS:
        if inst not in tr["pos"]:
            continue
        k = tr["kin"][inst]
        row[f"{inst}_inuse_pct"] = round(100 * float(tr["inuse_frac"][inst]), 2)
        # tracking success over in-use frames only (parked frames don't need tracking)
        row[f"{inst}_tracked_pct"] = _r(100 * tr["track_ok_inuse"][inst], 2)
        row[f"{inst}_resect_dist_mm"] = _r(tr["resection_dist_mean"][inst], 2)
        row[f"{inst}_path_mm"] = _r(tr["pathlen"][inst], 1)
        row[f"{inst}_speed_mean"] = _r(_nanmean(k["velocity"]), 2)
        row[f"{inst}_speed_peak"] = _r(_nanmax(k["velocity"]), 2)
        row[f"{inst}_accel_mean"] = _r(_nanmean(k["acceleration"]), 2)
        row[f"{inst}_jerk_mean"] = _r(_nanmean(k["jerk"]), 1)
        row[f"{inst}_straightness"] = _r(tr["straightness"][inst], 3)
        row[f"{inst}_workvol_mm3"] = _r(tr["bbox_vol"][inst], 0)
        row[f"{inst}_ellvol_mm3"] = _r(tr["ell_vol"][inst], 0)
        row[f"{inst}_aniso"] = _r(tr["aniso"][inst], 2)
        row[f"{inst}_occ_eff"] = _r(tr["occ_eff"][inst], 1)
        row[f"{inst}_idle_frac"] = _r(tr["idle_frac"][inst], 3)
        row[f"{inst}_angspeed_dps"] = _r(_nanmean(tr["ang_speed"][inst]), 2)
        row[f"{inst}_angaccel_dps2"] = _r(_nanmean(np.abs(tr["ang_accel"][inst])), 2)
    for pair in PAIR_KEYS:
        if pair in tr["dist"]:                          # distance uses the §5 in-use mask (NaN elsewhere)
            d = tr["dist"][pair]
            valid = np.any(~np.isnan(d))
            row[f"dist_{pair}_mean"] = round(float(np.nanmean(d)), 1) if valid else np.nan
            row[f"dist_{pair}_min"] = round(float(np.nanmin(d)), 1) if valid else np.nan
            row[f"dist_{pair}_max"] = round(float(np.nanmax(d)), 1) if valid else np.nan
        if pair in tr["angle"]:
            a = tr["angle"][pair]
            row[f"angle_{pair}_mean"] = (round(float(np.nanmean(a)), 1)
                                         if np.any(~np.isnan(a)) else np.nan)
    rows.append(row)

per_trial = pd.DataFrame(rows)

# per-participant aggregate: mean of the numeric per-trial metrics
num_cols = [c for c in per_trial.columns if c not in ("participant", "trial", "file", "n_frames")]
per_participant = (per_trial.groupby("participant")[num_cols]
                   .mean().round(3).reset_index())
per_participant.insert(1, "n_trials",
                       per_trial.groupby("participant").size().values)

# save next to the zip
OUT_DIR = os.path.dirname(os.path.abspath(DATA_ZIP))
out_trial = os.path.join(OUT_DIR, "metrics_per_trial.csv")
out_part = os.path.join(OUT_DIR, "metrics_per_participant.csv")
per_trial.to_csv(out_trial, index=False)
per_participant.to_csv(out_part, index=False)
print("wrote", out_trial, "and", out_part)

## Export — single JSON for the visualization notebook

Everything computed above (per-trial time series, per-instrument and per-pair scalar
metrics, and the statistics tables) is written to **one JSON file**, `DATA_JSON`. The
visualization notebook reads only this file. `NaN` values (masked / not-in-use frames)
are preserved and read back correctly by Python's `json` module.

The export also carries what the force pyramid needs to be built on the resection site:
each trial's registered **`resection`** point (the center of its pyramid map) and the
shared **`plane_axes`** `(u, v, n)` of the fiducial working plane (`n` = out-of-plane
normal, the depth axis the force is summed along).

In [ ]:
import json

# ---- Export everything needed for plotting into a single JSON --------------------
DATA_JSON = os.path.join(os.path.dirname(os.path.abspath(DATA_ZIP)), "analysis_data.json")

def _arr(a):
    "numpy array (any shape) -> nested python lists (NaN preserved as JSON NaN token)."
    return np.asarray(a, float).tolist()

def _scal(x):
    "scalar -> python float (NaN preserved)."
    return float(x)

trials_json = []
for tr in trials:
    insts = list(tr["pos"].keys())            # instruments present in this trial
    tj = {
        "participant": tr["participant"],
        "trial": tr["trial"],
        "name": tr["name"],
        "label": tr["label"],
        "n_frames": int(len(tr["timestamps"])),
        "duration": _scal(tr["timestamps"][-1] - tr["timestamps"][0]),
        "rmse": _scal(tr["rmse"]),
        "present_instruments": insts,
        # fixed resection point of this trial, in the common registered frame — the center
        # of its force-pyramid map (None when the recording carries no Resection_Location)
        "resection": (_arr(tr["resection_reg"]) if tr["resection_reg"] is not None else None),
        # ---- time-series signals ----
        "tnorm": _arr(tr["tnorm"]),
        "fmag": _arr(tr["fmag"]),
        "dFdt": _arr(tr["dFdt"]),
        "kin": {inst: {m: _arr(tr["kin"][inst][m]) for m in ("velocity", "acceleration", "jerk")}
                for inst in insts},
        "pos_plot": {inst: _arr(tr["pos_plot"][inst]) for inst in insts},
        "ang_speed": {inst: _arr(tr["ang_speed"][inst]) for inst in tr["ang_speed"]},
        "ang_accel": {inst: _arr(tr["ang_accel"][inst]) for inst in tr["ang_accel"]},
        "tracking_status": {inst: [int(v) for v in tr["tracking_status"][inst]]
                            for inst in tr["tracking_status"]},
        "in_use": {inst: [int(v) for v in tr["in_use"][inst]] for inst in insts},
        "resection_dist": {inst: _arr(tr["resection_dist"][inst]) for inst in insts},
        "dist": {pair: _arr(tr["dist"][pair]) for pair in tr["dist"]},
        "angle": {pair: _arr(tr["angle"][pair]) for pair in tr["angle"]},
        # ---- per-trial scalar force metrics ----
        "force_mean": _scal(tr["fmag"].mean()),
        "force_peak": _scal(tr["fmag"].max()),
        "force_cov": _scal(tr["force_cov"]),
        "impulse": _scal(tr["impulse"]),
        "dFdt_abs_mean": _scal(tr["dFdt_abs_mean"]),
        "torque_mean": _scal(tr["tmag"].mean()),
        # ---- per-instrument scalar geometry / economy / localization metrics ----
        "inuse_frac": {inst: _scal(tr["inuse_frac"][inst]) for inst in insts},
        # tracking success over in-use frames only
        "track_ok_inuse": {inst: _scal(tr["track_ok_inuse"][inst]) for inst in insts},
        "resection_dist_mean": {inst: _scal(tr["resection_dist_mean"][inst]) for inst in insts},
        "pathlen": {inst: _scal(tr["pathlen"][inst]) for inst in insts},
        "netdisp": {inst: _scal(tr["netdisp"][inst]) for inst in insts},
        "straightness": {inst: _scal(tr["straightness"][inst]) for inst in insts},
        "bbox_vol": {inst: _scal(tr["bbox_vol"][inst]) for inst in insts},
        "idle_frac": {inst: _scal(tr["idle_frac"][inst]) for inst in insts},
        "ell_vol": {inst: _scal(tr["ell_vol"][inst]) for inst in insts},
        "aniso": {inst: _scal(tr["aniso"][inst]) for inst in insts},
        "occ_eff": {inst: _scal(tr["occ_eff"][inst]) for inst in insts},
        "occ_voxels": {inst: int(tr["occ_voxels"][inst]) for inst in insts},
    }
    trials_json.append(tj)

payload = {
    "meta": {
        "instruments": INSTRUMENTS,
        "pairs": [list(p) for p in PAIRS],
        "pair_keys": PAIR_KEYS,
        "participants": list(participants.keys()),
        "smooth_window": SMOOTH_WINDOW,
        "idle_speed": IDLE_SPEED,
        "voxel_size": VOXEL_SIZE,
        "resection_dist_max": RESECTION_DIST_MAX,
        "time_unit": TIME_UNIT,
        # 4 registration fiducials in the common frame (the working-area quadrilateral
        # every trial is registered onto); used to scale the force-pyramid plots.
        "fiducials": (_arr(ref_pts) if ref_pts is not None else None),
        # orthonormal axes of the fiducial working plane, in the common frame:
        # [u, v, n] with u, v in plane and n the out-of-plane normal (the pyramid's
        # depth axis). The pyramid's origin is each trial's own "resection" point.
        "plane_axes": (None if PLANE_N is None
                       else [_arr(PLANE_U), _arr(PLANE_V), _arr(PLANE_N)]),
    },
    "trials": trials_json,
    "tables": {
        "per_trial": per_trial.to_dict(orient="records"),
        "per_participant": per_participant.to_dict(orient="records"),
    },
}

with open(DATA_JSON, "w") as fh:
    json.dump(payload, fh)      # allow_nan=True (default): NaN is written and read back by Python
print(f"wrote {DATA_JSON}  ({os.path.getsize(DATA_JSON) / 1e6:.2f} MB, "
      f"{len(trials_json)} trials)")

---
### Notes & assumptions

- **Data layout.** One zip (`DATA_ZIP`) containing `<participant>/<trial>.igs.mha`
  subfolders; it is extracted into `WORK_DIR` on every run. Participants are the
  folders that directly contain the `.igs.mha` files (a wrapper folder in the zip is
  ignored); trials are numbered by filename order within each participant folder.
- **Single reference frame.** All trials of all participants are registered onto
  `trials[0]`'s fiducials, so everything is comparable. Change which trial is the
  anchor by reordering, or set `ref_pts` yourself in §4.
- **Fiducial order matters.** Registration assumes `BipolarCollectedPoint0..3`
  correspond across trials; a high RMSE in §4 usually means a swapped/mis-collected point.
- **Tracking status.** Read from the per-frame `Seq_Frame<idx>_<Instrument>Status` field
  (`OK` / `INVALID`), or `…TipToWorldTransformStatus` when that is the name used; it
  becomes `tr["tracking_status"][inst]` (1 = tracked, 0 = not tracked). Recordings with no
  status field at all fall back to the legacy frozen-pose heuristic (a 4×4 pose bit-for-bit
  identical to the previous frame). **Tracking success is only meaningful while an
  instrument is in use**, so it is always reported over in-use frames
  (`tr["track_ok_inuse"][inst]`, and the `<inst>_tracked_pct` table column).
- **In-use masking.** `Resection_Location` (trial header) is a fixed point for the whole
  trial. An instrument is **in use** on frames where its tip is within
  `RESECTION_DIST_MAX = 30 mm` of it (`tr["in_use"][inst]`, share reported as
  `tr["inuse_frac"][inst]`); farther away it is parked and ignored. The mask applied to the
  signals, `tr["valid"][inst]`, is that flag **and** valid tracking, since an `INVALID`
  frame carries no usable pose — **every tracking-derived signal is set to NaN outside it**.
  Velocities are still differentiated across the full track first, then masked, so gaps
  don't corrupt the derivatives. If a recording has no `Resection_Location`, the proximity
  gate is skipped (all frames in use) and a warning is printed in §3.
- **The fiducial plane is orientation only.** It is no longer an in-use gate; it survives
  as the frame the force pyramid is drawn on — in-plane axes `u, v` and the out-of-plane
  normal `n` (§4, exported as `meta["plane_axes"]`), with the pyramid centered on the
  trial's registered `Resection_Location`.
- **Smoothing.** `SMOOTH_WINDOW` sets the Savitzky-Golay window used before
  differentiation. Larger = smoother but more lag; `0` differentiates the raw track.
- **Force is a single sensor**, not per-instrument, so it is summarized per
  participant/trial. Torque magnitude is also parsed (`tr["tmag"]`) if you want it.
- **Inter-instrument distance** is computed only for pairs used together
  (Bipolar–Cavitron, Bipolar–Scissors; Cavitron–Scissors is excluded). It is masked by
  the §5 in-use flag — frames where either instrument is not in use are set to NaN and
  dropped from plots and averages. It is computed on registered positions, but a rigid
  transform preserves distance, so registration does not change it.
- **Instrument long axis / orientation.** The pivot-calibrated `*TipToWorldTransform`
  puts the tip at the frame origin; the shaft runs along the pivot direction
  `â = normalize(−Rᵀd)` (from the constant `*TipTo*Transform`), **not** along a tip-frame
  coordinate axis. The inter-instrument **angle** is between these axes (0°=parallel,
  180°=anti-parallel), NaN where either instrument is untracked. **Angular speed** is
  the rotational speed of the instrument frame (deg/s). If your tools had a full spin
  calibration you could instead use a fixed tip-frame axis — swap `long_axis_tip`.
- **Suggested metrics.** `straightness = net displacement / path length` is a simple
  economy-of-motion measure — note it drops toward 0 for paths that loop back near
  their start even when efficient. `working volume` is the axis-aligned bounding-box
  volume (swap for a convex-hull volume via `scipy.spatial.ConvexHull` if you prefer).
  `idle fraction` uses the `IDLE_SPEED` threshold (mm/s) — tune it to your setup.
  Other metrics worth adding if useful: **submovement count** (velocity peaks, a
  smoothness proxy), **spectral arc length / dimensionless jerk** (established movement-
  smoothness scores), and **speed correlation** between the two instruments (bimanual
  coordination).